# CPG Training: Fine-tuning T5-base Paraphraser on PAWS + QQP

This notebook fine-tunes `humarin/chatgpt_paraphraser_on_T5_base` (220M params) on:
- **PAWS** (labeled_final, label=1 pairs) — adversarial paraphrase pairs
- **QQP** (from GLUE, label=1 pairs) — Quora duplicate question pairs

**Runtime:** Google Colab with T4 GPU (free tier)

**Why this base model?** Raw t5-small produces near-copies (Self-BLEU=94). This model
is already trained on ChatGPT-generated paraphrases and produces diverse rewrites.
Fine-tuning further on PAWS+QQP customizes it for our use case.

In [2]:
# Install dependencies
!pip install -q transformers datasets accelerate sentencepiece torch

In [3]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU: Tesla T4
Memory: 15.6 GB


## 1. Load Base Model and Tokenizer

In [4]:
MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters: 222.9M


## 2. Load and Prepare Training Data

We use positive pairs (label=1) from both datasets:
- PAWS: sentence pairs that are paraphrases
- QQP: Quora question pairs that are duplicates

In [5]:
# Load PAWS — labeled_final split, label=1 (paraphrase pairs)
paws = load_dataset("paws", "labeled_final", split="train")
paws = paws.filter(lambda x: x["label"] == 1)
print(f"PAWS positive pairs: {len(paws)}")

# Rename columns to standard format
paws = paws.rename_column("sentence1", "input_text")
paws = paws.rename_column("sentence2", "target_text")
paws = paws.remove_columns(["label", "id"])

README.md: 0.00B [00:00, ?B/s]

labeled_final/train-00000-of-00001.parqu(…):   0%|          | 0.00/8.43M [00:00<?, ?B/s]

labeled_final/test-00000-of-00001.parque(…):   0%|          | 0.00/1.24M [00:00<?, ?B/s]

labeled_final/validation-00000-of-00001.(…):   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49401 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/49401 [00:00<?, ? examples/s]

PAWS positive pairs: 21829


In [6]:
# Load QQP from GLUE — label=1 (duplicate question pairs)
qqp = load_dataset("glue", "qqp", split="train")
qqp = qqp.filter(lambda x: x["label"] == 1)
print(f"QQP positive pairs: {len(qqp)}")

# Rename columns
qqp = qqp.rename_column("question1", "input_text")
qqp = qqp.rename_column("question2", "target_text")
qqp = qqp.remove_columns(["label", "idx"])

README.md: 0.00B [00:00, ?B/s]

qqp/train-00000-of-00001.parquet:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

qqp/validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

Filter:   0%|          | 0/363846 [00:00<?, ? examples/s]

QQP positive pairs: 134378


In [7]:
# Combine datasets
# Use a subset for faster training (full dataset is large)
MAX_SAMPLES = 50000  # 50K total — enough for fine-tuning, fits in Colab time

paws_subset = paws.shuffle(seed=42).select(range(min(len(paws), MAX_SAMPLES // 2)))
qqp_subset = qqp.shuffle(seed=42).select(range(min(len(qqp), MAX_SAMPLES // 2)))

combined = concatenate_datasets([paws_subset, qqp_subset]).shuffle(seed=42)
print(f"Combined training set: {len(combined)} pairs")
print(f"Sample: {combined[0]}")

Combined training set: 46829 pairs
Sample: {'input_text': 'In how many days can I get a Dubai visa online?', 'target_text': 'How many days are required to get a Dubai work visa online?'}


## 3. Tokenize the Dataset

In [8]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 256

def tokenize_function(examples):
    """Tokenize input-target pairs with 'paraphrase: ' prefix."""
    inputs = [f"paraphrase: {text}" for text in examples["input_text"]]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding=False
    )

    labels = tokenizer(
        targets, max_length=MAX_TARGET_LENGTH, truncation=True, padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = combined.map(
    tokenize_function,
    batched=True,
    remove_columns=combined.column_names,
    desc="Tokenizing",
)

# Train/val split
split = tokenized.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Tokenizing:   0%|          | 0/46829 [00:00<?, ? examples/s]

Train: 44487, Val: 2342


## 4. Training Configuration

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./cpg-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

print("Trainer ready. Starting fine-tuning...")

Trainer ready. Starting fine-tuning...


In [10]:
# Train!
trainer.train()

Step,Training Loss,Validation Loss
500,0.952182,0.820425
1000,0.849007,0.774502
1500,0.884682,0.746173
2000,0.832332,0.739730
2500,0.816660,0.718230
3000,0.749653,0.705497
3500,0.794844,0.699712
4000,0.794675,0.691785
4500,0.744053,0.680258
5000,0.786257,0.671942


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=16683, training_loss=0.6372252191448572, metrics={'train_runtime': 5147.8211, 'train_samples_per_second': 25.926, 'train_steps_per_second': 3.241, 'total_flos': 6951322114882560.0, 'train_loss': 0.6372252191448572, 'epoch': 3.0})

## 5. Save the Fine-tuned Model

In [11]:
# Save locally
SAVE_PATH = "./cpg-finetuned-final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./cpg-finetuned-final


In [12]:
# Optional: Push to HuggingFace Hub
# from huggingface_hub import login
# login()
# trainer.push_to_hub("your-username/cpg-paraphraser")

## 6. Quick Test of Fine-tuned Model

In [13]:
# Load fine-tuned model for inference
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

ft_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
ft_model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_PATH).to("cuda")
ft_model.eval()

test_sentence = "A cover letter is a formal document that accompanies your resume when you apply for a job."

inputs = ft_tokenizer(
    f"paraphrase: {test_sentence}",
    return_tensors="pt",
    max_length=256,
    truncation=True,
).to("cuda")

with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        do_sample=True,
        temperature=1.5,
        top_k=120,
        top_p=0.95,
        no_repeat_ngram_size=2,
        num_return_sequences=5,
        max_length=256,
    )

print("Input:", test_sentence)
print("\nGenerated candidates:")
for i, out in enumerate(outputs):
    decoded = ft_tokenizer.decode(out, skip_special_tokens=True)
    print(f"  [{i+1}] {decoded}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Input: A cover letter is a formal document that accompanies your resume when you apply for a job.

Generated candidates:
  [1] Cover letter is a formal document that goes with your resume when you attempt for 's candidate - or not when they file for the job.
  [2] A catch letter is the formal document that goes along with your resume when you apply for job in any market.
  [3] Când you apply for job, a cover letter is t . formal document that accompanies your Resume :
  [4] A cover letter is a formal document that comes with your resume when you apply to work..
  [5] A cover letter is a formal document accompanying your resume when you apply for , or are in, certain career fields (including the digital transformation of digital identity into nervos).)


In [14]:
# Download the fine-tuned model (for local use)
!zip -r cpg-finetuned-final.zip cpg-finetuned-final/
from google.colab import files
files.download("cpg-finetuned-final.zip")

  adding: cpg-finetuned-final/ (stored 0%)
  adding: cpg-finetuned-final/training_args.bin (deflated 53%)
  adding: cpg-finetuned-final/model.safetensors (deflated 7%)
  adding: cpg-finetuned-final/tokenizer.json (deflated 79%)
  adding: cpg-finetuned-final/config.json (deflated 65%)
  adding: cpg-finetuned-final/tokenizer_config.json (deflated 83%)
  adding: cpg-finetuned-final/generation_config.json (deflated 29%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>